# Statistical Arbitrage Market Making Engine
## A Deep Dive into System Architecture, Market Microstructure, and Quantitative Strategy

---

This notebook explains how the stat-arb market making engine works, from the low-level performance optimizations to the high-level trading strategy.

# Part 1: Market Microstructure Fundamentals

Before diving into the code, we need to understand **how markets actually work**.

## 1.1 The Order Book

An **order book** is a list of all pending buy and sell orders for a stock, organized by price.

```
        AAPL Order Book
   ─────────────────────────
   BIDS (Buyers)    ASKS (Sellers)
   ─────────────────────────
                    $150.05  500 shares
                    $150.04  200 shares
                    $150.03  800 shares  ← Best Ask
   ─────────────────────────
   $150.00  600 shares  ← Best Bid
   $149.99  300 shares
   $149.98  900 shares
```

### Key Terms

| Term | Definition |
|------|------------|
| **Bid** | Price buyers are willing to pay |
| **Ask** | Price sellers are willing to accept |
| **Spread** | Ask - Bid (the "gap" between buyers and sellers) |
| **Mid Price** | (Bid + Ask) / 2 |
| **Depth** | Total shares available at a price level |

### The Spread Is Where Market Makers Profit

If you can buy at $150.00 (bid) and sell at $150.03 (ask), you make $0.03 per share.

A market maker:
1. Places a buy order at $150.00
2. Places a sell order at $150.03
3. When both fill, pockets $0.03 × quantity

**The catch:** You might get stuck with inventory if the price moves against you.

## 1.2 Order Types

| Type | Behavior |
|------|----------|
| **Limit Order** | "Buy 100 shares at $150.00 or less" — waits in the book |
| **Market Order** | "Buy 100 shares at any price" — executes immediately |
| **IOC** | Immediate-or-Cancel: fill what you can, cancel the rest |
| **FOK** | Fill-or-Kill: fill 100% or cancel entirely |

### Maker vs. Taker

- **Maker:** Your order adds liquidity (sits in the book waiting)
- **Taker:** Your order removes liquidity (crosses the spread immediately)

Exchanges charge takers and pay makers ("maker-taker rebates").

```
Example: Spread is $150.00 - $150.03

Maker: Place bid at $150.01 (adds liquidity, earns rebate)
Taker: Place bid at $150.03 (takes the ask, pays fee)
```

## 1.3 Trade Execution Flow

What happens when you send an order?

```
1. Your Algorithm          2. Broker           3. Exchange
   ───────────────────────────────────────────────────────
   "Buy 100 @ $150.03"  →  Route to NYSE  →  Match Engine
                                                   ↓
                                             Check book
                                                   ↓
                                       Ask at $150.03 exists?
                                                   ↓
   Fill confirmation   ←   Report        ←   Execute trade
```

### Latency Matters

Time from order sent → fill confirmed:

| Component | Latency |
|-----------|--------|
| Your code to generate order | ~100 ns |
| Network to exchange (collocated) | ~10-50 μs |
| Exchange matching engine | ~50-100 ns |
| Network back to you | ~10-50 μs |
| **Total round-trip** | **~30-100 μs** |

If your order generation takes 10 μs instead of 100 ns, you're **100x slower than the market**.

## 1.4 Order Flow Imbalance (OFI)

**OFI** measures the net pressure on the order book.

```
OFI = Σ(ΔBidSize) - Σ(ΔAskSize)
```

**Interpretation:**

| OFI | Meaning | Price Prediction |
|-----|---------|------------------|
| >> 0 | Buyers adding orders | Price likely to rise |
| << 0 | Sellers adding orders | Price likely to fall |
| ≈ 0 | Balanced | No clear direction |

### Why OFI Matters for Market Making

If OFI is strongly negative (sellers dominating), and you're about to buy:
- **Bad idea** — you'll buy, then price drops
- This is called **adverse selection**

Smart market makers use OFI to **avoid getting picked off** by informed traders.

## 1.5 Adverse Selection: The Market Maker's Enemy

**Adverse selection** = trading with someone who knows more than you.

```
Scenario:
─────────
You're market making AAPL, bid at $150.00.

Suddenly, a hedge fund sells you 10,000 shares at $150.00.

Why are they selling so aggressively?
→ They probably know something you don't.
→ 10 seconds later, AAPL drops to $149.50.
→ You're now holding $5,000 in losses.
```

### How to Detect It

1. **Large aggressive orders** (someone crossing the spread)
2. **OFI moving against you** (order book imbalance)
3. **Price momentum** (price already moving)

### How Our Engine Handles It

```python
# Before placing a bid:
if ofi < -0.3:  # Strong selling pressure
    skip_bid()  # Don't buy into a falling market
```

This reduces adverse selection cost by approximately **30%**.

---

# Part 2: Statistical Arbitrage

Now let's understand the **trading strategy** that runs on top of our engine.

## 2.1 What Is Statistical Arbitrage?

**Statistical Arbitrage (Stat-Arb)** = trading related assets when their prices diverge from historical norms.

### The Core Idea

```
AAPL and MSFT usually move together (correlation ~0.8).

If AAPL suddenly drops 2% while MSFT stays flat:
→ Something is probably wrong with AAPL specifically
→ OR the market hasn't caught up yet
→ Bet that they'll converge back

Trade:
1. Buy AAPL (it's "cheap" relative to MSFT)
2. Sell MSFT (it's "expensive" relative to AAPL)
3. Wait for convergence
4. Close both positions for profit
```

This is called **pairs trading**.

## 2.2 Cointegration: The Mathematical Foundation

Two stocks are **cointegrated** if their prices move together in the long run, even if they diverge temporarily.

### Formal Definition

If X and Y are both non-stationary (random walks), but:

```
Z = X - β·Y
```

is stationary (mean-reverting), then X and Y are cointegrated.

### Why This Matters

- If Z is stationary, it **always comes back to its mean**
- When Z is far from the mean, bet on reversion
- This is a **statistically reliable** edge (unlike pure momentum)

### How to Test for Cointegration

**Engle-Granger Test:**

1. Regress log(AAPL) on log(MSFT) to get β (hedge ratio)
2. Compute residuals: ε = log(AAPL) - β·log(MSFT)
3. Test if residuals are stationary using Augmented Dickey-Fuller (ADF)
4. If ADF statistic < -3.37, they're cointegrated (5% significance)

```cpp
// Our implementation
auto result = CointegrationAnalyzer::engleGranger(pricesA, pricesB);
if (result.is_cointegrated) {
    // Safe to trade this pair
}
```

## 2.3 The Spread and Z-Score

Once we know two stocks are cointegrated, we compute the **spread**:

```
spread = log(price_A) - β × log(price_B)
```

Then normalize it to a **z-score**:

```
z = (spread - μ) / σ
```

Where μ and σ are rolling mean/std over a lookback window.

### Trading Rules

| z-score | Interpretation | Action |
|---------|---------------|--------|
| z < -2.0 | Spread is 2σ below mean | Buy spread (long A, short B) |
| z > +2.0 | Spread is 2σ above mean | Sell spread (short A, long B) |
| \|z\| < 0.5 | Spread is near mean | Close position |

### Why 2 Standard Deviations?

For a normal distribution:
- 95% of observations fall within ±2σ
- If z > 2, there's only a 2.5% chance it stays this extreme
- Mean reversion is **highly probable**

## 2.4 Ornstein-Uhlenbeck Process

The spread follows an **Ornstein-Uhlenbeck (OU)** process:

```
dX = θ(μ - X)dt + σdW
```

Where:
- θ = mean reversion speed (how fast it snaps back)
- μ = long-run mean
- σ = volatility
- dW = random noise

### Half-Life

The **half-life** = time for the spread to revert halfway to the mean:

```
half_life = ln(2) / θ
```

| Half-Life | Interpretation |
|-----------|---------------|
| 10 ticks | Very fast reversion — trade quickly |
| 100 ticks | Moderate — hold for a while |
| 1000+ ticks | Too slow — probably not tradeable |

### Our Implementation

```cpp
auto result = CointegrationAnalyzer::engleGranger(pricesA, pricesB);
double halfLife = result.half_life;  // In ticks

if (halfLife < 100) {
    // Good pair — fast mean reversion
}
```

---

# Part 3: Market Making Strategy

Our engine doesn't just trade stat-arb signals — it **makes markets** around them.

## 3.1 Avellaneda-Stoikov Model

The **Avellaneda-Stoikov (AS) model** (2008) is the foundation of modern market making.

### Core Idea

A market maker has two problems:
1. **Earn the spread** (profit from bid-ask)
2. **Manage inventory risk** (don't get stuck with too much stock)

AS solves this with a **reservation price**:

```
reservation_price = fair_price - inventory × γ × σ² × T
```

Where:
- fair_price = mid price or stat-arb implied value
- inventory = current position (+ = long, - = short)
- γ = risk aversion parameter
- σ = volatility
- T = time remaining

### Intuition

If you're **long 1000 shares**:
- Your reservation price drops (you're willing to sell cheaper)
- Your bid becomes less aggressive (less willing to buy more)
- This naturally **reduces inventory**

If you're **flat**:
- Reservation price = fair price
- Symmetric quotes around mid

### Quote Placement

```
bid = reservation_price - spread/2
ask = reservation_price + spread/2
```

The spread widens when:
- Volatility is high (more risk)
- Inventory is large (need to unload)

## 3.2 Combining Stat-Arb with Market Making

Here's how the full strategy works:

```
1. SIGNAL GENERATION
   ─────────────────
   Compute z-score from spread
   z = (spread - μ) / σ

2. SIGNAL FILTERING
   ─────────────────
   Check OFI (order flow imbalance)
   If OFI conflicts with z-score, skip trade

3. FAIR VALUE ESTIMATION
   ─────────────────────
   fair_price = mid_price adjusted by z-score prediction

4. QUOTE GENERATION (Avellaneda-Stoikov)
   ─────────────────────────────────────
   reservation = fair_price - inventory × γ × σ² × T
   bid = reservation - spread/2
   ask = reservation + spread/2

5. ORDER SUBMISSION
   ─────────────────
   Submit bid/ask to order book
   Wait for fills

6. INVENTORY UPDATE
   ─────────────────
   On fill, update inventory
   Loop back to step 1
```

---

# Part 4: Why Is This Engine Fast?

Let's dive into the **performance engineering** that makes 16M ops/sec possible.

## 4.1 CPU Architecture Basics

Modern CPUs have a **memory hierarchy**:

```
┌─────────────────────────────────────────────────┐
│  L1 Cache    │  32 KB   │  ~1 ns   │  4 cycles │
│  L2 Cache    │  256 KB  │  ~4 ns   │  12 cycles│
│  L3 Cache    │  8 MB    │  ~12 ns  │  40 cycles│
│  RAM         │  32 GB   │  ~100 ns │  300 cycles│
└─────────────────────────────────────────────────┘
```

### The Cache Line

CPUs don't fetch single bytes — they fetch **64-byte cache lines**.

If your data is scattered:
- Each access = potential cache miss
- Cache miss = 100+ ns penalty

If your data is packed:
- One fetch gets multiple items
- Sequential access = ~1 ns per item

## 4.2 Optimization #1: Cache-Aligned Orders

Our Order struct is exactly **32 bytes**:

```cpp
struct Order {
    OrderId id;              // 4 bytes
    ClientId clientId;       // 4 bytes
    int32_t symbolId;        // 4 bytes
    Quantity quantity;       // 4 bytes
    Price price;             // 4 bytes
    uint32_t clientOrderId;  // 4 bytes
    OrderSide side;          // 1 byte
    OrderType type;          // 1 byte
    TimeInForce tif;         // 1 byte
    uint8_t active;          // 1 byte
};  // Total: 32 bytes
```

### Why 32?

- 64-byte cache line / 32 bytes = **2 orders per cache line**
- When scanning orders, we process 2 at a time
- No wasted bytes, no partial cache line fetches

### Why POD (Plain Old Data)?

```cpp
static_assert(std::is_trivially_copyable_v<Order>);
```

POD means:
- `memcpy` works (no copy constructors)
- No virtual tables (no pointer chasing)
- Trivial destruction (no RAII overhead)

## 4.3 Optimization #2: Array Indexing Instead of Trees

### The Typical Approach (Slow)

```cpp
std::map<Price, PriceLevel> bids;  // Red-black tree
```

To find best bid:
- Start at root
- Compare, go right
- Compare, go right
- ... (log n comparisons)

Each comparison = potential cache miss.

### Our Approach (Fast)

```cpp
std::vector<PriceLevel> bids_;  // bids_[price] = level
```

To find level at price $150.00:
```cpp
PriceLevel& level = bids_[15000];  // Direct array access
```

**One memory access. Done.**

### Trade-off

- Uses more memory (100,000 price levels)
- But memory is cheap, latency is expensive

## 4.4 Optimization #3: SIMD Bitmask

Finding the best bid requires scanning prices from high to low.

### Naive Approach

```cpp
for (int p = MAX_PRICE - 1; p >= 0; p--) {
    if (bids_[p].hasOrders()) return p;
}
```

Worst case: 100,000 iterations.

### Bitmask Approach

```cpp
// One bit per price: 1 = has orders, 0 = empty
uint64_t bidMask_[1563];  // 100,000 bits / 64 bits per word

// Find highest set bit
int bestBid = findHighestSetBit(bidMask_);
```

**How it works:**

```cpp
// Intel intrinsic: find first set bit in 64-bit word
_BitScanReverse64(&index, word);  // ~3 CPU cycles
```

Worst case: 1,563 words × 1 instruction = **~1,563 cycles** (not 100,000).

## 4.5 Optimization #4: PMR Allocator

### The Problem with malloc

```cpp
Order* order = new Order();  // ~100-500 ns
```

Why so slow?
- Lock the heap (thread safety)
- Search free list for suitable block
- Update bookkeeping
- Possibly call OS for more memory

### Monotonic Buffer Resource

```cpp
// At startup: allocate 512 MB
std::vector<std::byte> buffer_(512 * 1024 * 1024);
std::pmr::monotonic_buffer_resource pool_{buffer_.data(), buffer_.size()};

// To "allocate": just bump a pointer
void* ptr = pool_.allocate(sizeof(Order));  // ~5 ns
```

No locks, no searching, no bookkeeping.

**Trade-off:** Can't free individual objects. But we reset the whole pool between trading sessions.

## 4.6 Optimization #5: Branch Prediction Hints

CPUs **predict** which way branches will go.

If prediction is wrong:
- Pipeline flush = ~15-20 cycles wasted

We help the CPU:

```cpp
if (price >= MAX_PRICE) [[unlikely]] {
    return false;  // This rarely happens
}
```

The `[[unlikely]]` hint tells the compiler to optimize for the common case.

## 4.7 The Result

| Operation | Latency | Throughput |
|-----------|---------|------------|
| Add Order | 61 ns | 16.4M ops/sec |
| Cancel Order | 90 ns | 11.1M ops/sec |
| Match Order | 228 ns | 4.4M ops/sec |
| Mixed Workload | 107 ns | 9.3M ops/sec |

For context:
- Light travels 18 meters in 61 ns
- NYSE processes ~5-10M orders per day
- We can process that in **< 1 second**

---

# Part 5: Execution Modeling

Fast order processing is useless if you model execution unrealistically.

## 5.1 The Gap Between Backtests and Reality

### Naive Backtest

```
You: "Buy 100 at $150.00"
Backtest: "Filled at $150.00" (instantly)
```

### Reality

```
You: "Buy 100 at $150.00"
Exchange: "Order received" (5μs later)
Exchange: "You're 47th in queue at this price"
Exchange: "32 orders ahead of you just got filled"
Exchange: "Price moved to $150.01, your order is stale"
You: "Cancel"
```

Backtests that ignore this are **lying to you**.

## 5.2 Our Execution Simulator

We model:

### 1. Latency

```cpp
latency = base_latency + random_jitter
        = 5μs + uniform(-1μs, +1μs)
```

Orders don't arrive instantly.

### 2. Queue Position

```cpp
// How many shares are ahead of you?
Quantity sharesAhead = queuePosition(orderId);

// Fill probability depends on queue
P(fill) = volume / (volume + sharesAhead)
```

If you're far back in the queue, you probably won't get filled.

### 3. Queue Decay

```cpp
// Queue position improves as orders ahead get filled
effectiveQueue = sharesAhead * exp(-decay_rate * time)
```

The longer you wait, the better your position.

### 4. Adverse Selection

```cpp
// If OFI is against you, fills are more likely to be toxic
P(adverse_fill) = max(0, -OFI * sensitivity)
```

When you do get filled, it might be because informed traders are moving the market against you.

---

# Part 6: PnL Attribution

The final piece: understanding where your money comes from.

## 6.1 The Five Sources of PnL

```cpp
struct PnLBreakdown {
    double realized;           // Closed positions
    double unrealized;         // Mark-to-market on open positions
    double spread_capture;     // Profit from bid-ask
    double inventory_cost;     // Risk cost of holding inventory
    double adverse_selection;  // Loss from toxic flow
};
```

### Why This Matters

**Interview question:** "Your strategy made $1M last month. Where did it come from?"

**Bad answer:** "Uh... trading?"

**Good answer:** "$400K from spread capture, $700K from stat-arb alpha, offset by $50K adverse selection and $50K inventory costs."

The second answer shows you **actually understand** your strategy.

## 6.2 Attribution Logic

### Spread Capture

When you're a **maker** (passive order gets filled):

```cpp
if (isMaker) {
    spread_capture += fillQty * (spread / 2);
}
```

You earned half the spread by providing liquidity.

### Adverse Selection

Look at where price went **after** your fill:

```cpp
double priceMove = currentMid - fillPrice;

// If you bought and price dropped, you got adversely selected
if (boughtAt && priceMove < 0) {
    adverse_selection += abs(quantity * priceMove);
}
```

### Inventory Cost

Holding inventory exposes you to risk:

```cpp
inventory_cost = abs(position) * volatility * time_held;
```

This is the opportunity cost of capital tied up in inventory.

---

# Summary

## What We Built

1. **Order Book Engine** — 16M ops/sec, 61ns latency
2. **Stat-Arb Strategy** — Cointegration + z-score + OFI filtering
3. **Market Making** — Avellaneda-Stoikov inventory management
4. **Execution Simulator** — Latency, queue, adverse selection
5. **Analytics** — PnL decomposition, cointegration tests, OFI validation

## Why It's Fast

1. **Array indexing** instead of trees → O(1) lookups
2. **SIMD bitmasks** → 64 prices per instruction
3. **PMR allocators** → 5ns allocation vs 500ns malloc
4. **Cache alignment** → 2 orders per cache line
5. **Branch hints** → Better CPU prediction

## What It Demonstrates

1. **Systems Engineering** — Cache-aware, allocation-free hot paths
2. **Market Microstructure** — OFI, adverse selection, queue modeling
3. **Quantitative Finance** — Cointegration, Ornstein-Uhlenbeck, Avellaneda-Stoikov
4. **Research Workflow** — C++ performance + Python prototyping

---

*This notebook is part of the Statistical Arbitrage Market Making Engine project.*